# Fine-Tune Qwen2.5-1.5B with MLX Distributed and LoRA

This Notebook will show how to fine-tune Qwen2.5-1.5B on multiple GPUs and `mlx-lm`.

The MLX runtime uses `mlx[cuda]` package to run distributed training on GPUs.

MLX Distributed: https://ml-explore.github.io/mlx/build/html/usage/distributed.html

MLX LM: https://github.com/ml-explore/mlx-lm

## Install the Kubeflow SDK

You need to install the Kubeflow SDK to interact with Kubeflow Trainer APIs:

In [ ]:
# !pip install -U kubeflow

## Create Script to Fine-Tune Qwen2.5

We will use `mlx-lm` library to fine-tune Qwen2.5.

`mlx-lm` is a Python package for generating text and fine-tuning LLMs with MLX.

We will perform LoRA (Low-Rank Adaptation) fine-tuning to reduce number of trainable parameters and optimize GPU resources.

In [ ]:
def fine_tune_qwen(num_samples: str, batch_size: str):
    import types
    import mlx.core as mx
    from mlx_lm.lora import train_model, CONFIG_DEFAULTS
    from mlx_lm.tuner.datasets import load_dataset
    from mlx_lm.utils import load
    from mlx_lm.generate import generate

    # Set parameters for the mlx-lm.
    args = types.SimpleNamespace()
    args.model = "Qwen/Qwen2.5-1.5B-Instruct"
    args.data = "mlx-community/WikiSQL"
    args.train = True
    # Configure LoRA settings to reduce number of trainable params.
    args.lora_parameters = {
        "rank": 8,
        "dropout": 0.05,
        "scale": 20.0,
    }

    args.iters = int(num_samples)
    args.batch_size = int(batch_size)

    # Set defaults for other required parameters
    for k, v in CONFIG_DEFAULTS.items():
        if not hasattr(args, k):
            setattr(args, k, v)

    model, tokenizer = load(args.model)
    train_set, valid_set, test_set = load_dataset(args, tokenizer)

    # Start the Qwen distributed fine-tuning.
    train_model(args, model, train_set, valid_set)

    # Evaluate the fine-tuned adapter.
    dist = mx.distributed.init(strict=True, backend="mpi")
    if dist.rank() == 0:
        print("=" * 100)
        print(f"Training is complete. Adapters saved to: {args.adapter_path}")
        print("Evaluate the fine-tuned LoRA adapter for Qwen2.5")

        finetuned_model, finetuned_tokenizer = load(
            args.model, adapter_path=args.adapter_path
        )

        # Generate response using the fine-tuned adapter.
        sample_prompt = "What is SQL?"

        print(f"Prompt: {sample_prompt}")
        print("Response:")

        response = generate(
            model=finetuned_model,
            tokenizer=finetuned_tokenizer,
            prompt=sample_prompt,
            max_tokens=1000,
            verbose=False,
        )

        print(response)

## Get the MLX Runtime

You can list the available Kubeflow Trainer runtimes with the `list_runtimes()` API.

The name of the MLX runtime is `mlx-distributed`.

In [ ]:
from kubeflow.trainer import TrainerClient, CustomTrainer

for r in TrainerClient().list_runtimes():
    if r.name == "mlx-distributed":
        print(f"Name: {r.name}, Framework: {r.trainer.framework}, Trainer Type: {r.trainer.trainer_type.value}\n")
        mlx_runtime = r

## Get the Runtime Packages

You can see the available Python packages and GPUs with the `get_runtime_packages()` API.

The API shows available GPUs with CUDA driver on the single training node.

In [ ]:
TrainerClient().get_runtime_packages(mlx_runtime)

## Create TrainJob with MLX Distributed

Use the `train()` API to create distributed TrainJob on **4 GPUs**. Every MPI training node uses 1 GPU.

In [ ]:
job_id = TrainerClient().train(
    trainer=CustomTrainer(
        func=fine_tune_qwen,
        func_args={
            "num_samples": "100",
            # Batch size must be divisible by the number of GPUs. (8 / 4 = 2) per training node.
            "batch_size": "8",
        },
        num_nodes=4,  # Fine-Tune Qwen2.5 on 4 GPUs.
        resources_per_node={
            "gpu": 1
        },
    ),
    runtime=mlx_runtime,
)

In [ ]:
# Train API generates a random TrainJob id.
job_id

## Check the TrainJob Info

Use the `list_jobs()` and `get_job()` APIs to get information about created TrainJob and its steps.

In [ ]:
for job in TrainerClient().list_jobs():
    print(f"TrainJob: {job.name}, Status: {job.status}, Created at: {job.creation_timestamp}")

In [ ]:
# We execute mpirun command on node-0, which functions as the MPI Launcher node.
for c in TrainerClient().get_job(name=job_id).steps:
    print(f"Step: {c.name}, Status: {c.status}, Devices: {c.device} x {c.device_count}\n")

## Get the TrainJob Logs

Use the `get_job_logs()` API to retrieve the TrainJob logs.

The fine-tuning runs on 4 training nodes.

In [ ]:
for logline in TrainerClient().get_job_logs(job_id, follow=True):
    print(logline)

## Verify TrainJob Completion

Check the TrainJob status to ensure it completed successfully.

In [ ]:
TrainerClient().wait_for_job_status(name=job_id, timeout=1200)

## Delete the TrainJob

When TrainJob is finished, you can delete the resource.

In [ ]:
TrainerClient().delete_job(job_id)